# MindSync Stage 3 v2 — Leakage-free CMAF evaluation + JSD evaluation

**Why this notebook exists.** Stage 3 v1 sampled fusion *test* text from GoEmotions **train+validation**, which Stage 1 was fine-tuned on → the 94.83% is an upper bound. It also measured JSD on the CMAF auxiliary heads, not on the standalone posteriors the deployed app uses.

**What v2 does (Stage 1 & Stage 2 checkpoints unchanged):**
| Split | Text (GoEmotions) | Audio (RAVDESS) |
|---|---|---|
| train | `train` partition (balanced, 1500/cluster) | actors 1–18 |
| val | `validation` partition | actors 19–21 |
| test | **`test` partition (never seen by Stage 1)** | actors 22–24 |

1. Sanity check: standalone text/audio accuracy must reproduce 70.76% / ~40.56%.
2. Baselines on the **same** test pairs: text-only, audio-only, late fusion (avg probs).
3. CMAF full + ablation (concat, no aux heads, d=128, d=512) × **5 seeds** → mean ± std.
4. Bootstrap 95% CI + McNemar test (CMAF vs late fusion).
5. JSD on **standalone** posteriors (deployed) *and* aux heads, test actors only, ROC-AUC, alert-rule hit/false-alarm rates, threshold calibrated on val.

**Run:** Runtime → T4 GPU → add `HF_TOKEN` secret → Run all. At the end, copy the **SUMMARY** block and send it to Claude.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q transformers==4.45.2 datasets==2.21.0 librosa==0.10.2 soundfile scikit-learn scipy huggingface_hub==0.25.2

import os, json, time, random, pathlib, urllib.request, zipfile
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN'); assert HF_TOKEN, 'HF_TOKEN secret missing'
os.environ['HF_TOKEN'] = HF_TOKEN

ROOT = pathlib.Path('/content/drive/MyDrive/mindsync'); ROOT.mkdir(exist_ok=True)
EMB_DIR = ROOT / 'embeddings'; EMB_DIR.mkdir(exist_ok=True)
OUT_DIR = ROOT / 'stage3_v2'; OUT_DIR.mkdir(exist_ok=True)
device = torch.device('cuda')
print(torch.cuda.get_device_name(0))

## 1 · Cluster mappings (identical to Stage 1 / Stage 2 v1)

In [ ]:
CLUSTER_NAMES = ['Distress', 'Resilience', 'Aggression', 'Ambiguity']
C2I = {c: i for i, c in enumerate(CLUSTER_NAMES)}
EMOTION_TO_CLUSTER = {
    'grief': 'Distress', 'nervousness': 'Distress', 'fear': 'Distress', 'sadness': 'Distress', 'remorse': 'Distress',
    'joy': 'Resilience', 'admiration': 'Resilience', 'excitement': 'Resilience', 'relief': 'Resilience',
    'amusement': 'Resilience', 'approval': 'Resilience', 'curiosity': 'Resilience', 'desire': 'Resilience',
    'gratitude': 'Resilience', 'love': 'Resilience', 'optimism': 'Resilience', 'pride': 'Resilience',
    'realization': 'Resilience',
    'anger': 'Aggression', 'annoyance': 'Aggression', 'disgust': 'Aggression', 'disapproval': 'Aggression',
    'embarrassment': 'Aggression',
    'confusion': 'Ambiguity', 'disappointment': 'Ambiguity', 'surprise': 'Ambiguity', 'caring': 'Ambiguity',
    'neutral': 'Ambiguity',
}
RAVDESS_EMOTION_TO_CLUSTER = {1: 3, 2: 1, 3: 1, 4: 0, 5: 2, 6: 0, 7: 2, 8: 3}

## 2 · RAVDESS (reuses the Drive copy if present)

In [ ]:
DATA_DIR = ROOT / 'ravdess'; DATA_DIR.mkdir(exist_ok=True)
ZIP = DATA_DIR / 'ravdess.zip'
if not ZIP.exists() or ZIP.stat().st_size < 100_000_000:
    urllib.request.urlretrieve('https://zenodo.org/record/1188976/files/Audio_Speech_Actors_01-24.zip', ZIP)
AUDIO_ROOT = DATA_DIR / 'extracted'
if not AUDIO_ROOT.exists() or len(list(AUDIO_ROOT.rglob('*.wav'))) < 1000:
    AUDIO_ROOT.mkdir(exist_ok=True)
    with zipfile.ZipFile(ZIP) as z: z.extractall(AUDIO_ROOT)
wavs = sorted(AUDIO_ROOT.rglob('*.wav'))
print('RAVDESS files:', len(wavs))

## 3 · Stage 1 + Stage 2 models (same classes as v1) and checkpoints

In [ ]:
from transformers import RobertaModel, Wav2Vec2Model
from huggingface_hub import hf_hub_download

class MindSyncTextModel(nn.Module):
    def __init__(self, model_name='roberta-large', num_classes=4, dropout=0.1):
        super().__init__()
        self.encoder = nn.Module(); self.encoder.encoder = RobertaModel.from_pretrained(model_name)
        self.encoder.hidden_size = self.encoder.encoder.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Module(); h = self.encoder.hidden_size
        self.classifier.classifier = nn.Sequential(nn.Dropout(dropout), nn.Linear(h, h // 2), nn.GELU(),
                                                   nn.Dropout(dropout), nn.Linear(h // 2, num_classes))

class Wav2VecAudioEncoder(nn.Module):
    def __init__(self, model_name='facebook/wav2vec2-large-960h', dropout=0.1):
        super().__init__()
        self.encoder = Wav2Vec2Model.from_pretrained(model_name)
        self.hidden_size = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(dropout)
    def forward(self, iv):
        return self.dropout(self.encoder(input_values=iv).last_hidden_state.mean(dim=1))

class AudioClassificationHead(nn.Module):
    def __init__(self, hidden_size=1024, num_classes=4, dropout=0.1):
        super().__init__()
        self.classifier = nn.Sequential(nn.Dropout(dropout), nn.Linear(hidden_size, hidden_size // 2), nn.GELU(),
                                        nn.Dropout(dropout), nn.Linear(hidden_size // 2, num_classes))
    def forward(self, e): return self.classifier(e)

class MindSyncAudioModel(nn.Module):
    def __init__(self, model_name='facebook/wav2vec2-large-960h', num_classes=4):
        super().__init__()
        self.encoder = Wav2VecAudioEncoder(model_name)
        self.classifier = AudioClassificationHead(self.encoder.hidden_size, num_classes)

def load(model, path):
    sd = torch.load(path, map_location=device, weights_only=False)
    if isinstance(sd, dict) and 'model_state_dict' in sd: sd = sd['model_state_dict']
    model.load_state_dict(sd); model.eval()
    for p in model.parameters(): p.requires_grad = False
    return model

text_model  = load(MindSyncTextModel().to(device),  hf_hub_download('Ubaida1/mindsync-text-model',  'best_text_model.pt'))
audio_model = load(MindSyncAudioModel().to(device), hf_hub_download('Ubaida1/mindsync-audio-model', 'best_audio_model.pt'))
print('Stage 1 + Stage 2 loaded and frozen')

## 4 · Audio embeddings + standalone audio posteriors (same preprocessing as v1)

In [ ]:
import librosa
AUDIO_EMB_PATH = EMB_DIR / 'ravdess_embeddings.pt'   # same cache file as v1
if AUDIO_EMB_PATH.exists():
    c = torch.load(AUDIO_EMB_PATH, weights_only=False)
    audio_embs, audio_clusters, audio_actors = c['emb'], c['cluster'], c['actor']
else:
    audio_embs, audio_clusters, audio_actors = [], [], []
    for i, wp in enumerate(wavs):
        parts = wp.stem.split('-')
        if len(parts) != 7: continue
        emotion, actor = int(parts[2]), int(parts[6])
        wav, _ = librosa.load(str(wp), sr=16_000, mono=True)
        wav, _ = librosa.effects.trim(wav, top_db=30)
        if len(wav) > 80_000: s = (len(wav) - 80_000) // 2; wav = wav[s:s + 80_000]
        else: wav = np.pad(wav, (0, 80_000 - len(wav)))
        wav = wav / (np.abs(wav).max() + 1e-8)
        iv = torch.tensor(wav, dtype=torch.float32).unsqueeze(0).to(device)
        with torch.no_grad(), torch.cuda.amp.autocast():
            audio_embs.append(audio_model.encoder(iv).cpu().float().squeeze(0))
        audio_clusters.append(RAVDESS_EMOTION_TO_CLUSTER[emotion]); audio_actors.append(actor)
        if (i + 1) % 200 == 0: print(f'  {i+1}/{len(wavs)}', flush=True)
    audio_embs = torch.stack(audio_embs)
    audio_clusters = torch.tensor(audio_clusters); audio_actors = torch.tensor(audio_actors)
    torch.save({'emb': audio_embs, 'cluster': audio_clusters, 'actor': audio_actors}, AUDIO_EMB_PATH)

with torch.no_grad():
    audio_probs = F.softmax(audio_model.classifier(audio_embs.to(device)).float(), -1).cpu()

a_split = {'train': (audio_actors <= 18), 'val': (audio_actors >= 19) & (audio_actors <= 21), 'test': (audio_actors >= 22)}
for k, m in a_split.items():
    acc = (audio_probs[m].argmax(-1) == audio_clusters[m]).float().mean().item()
    print(f'audio {k:5s}: {int(m.sum()):4d} clips | standalone acc {acc:.4f}')
print('→ test acc should be close to the Stage 2 v1 figure (0.4056)')

## 5 · Text embeddings per GoEmotions partition + standalone text posteriors

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer
TEXT_EMB_PATH = EMB_DIR / 'goemotions_split_embeddings_v3.pt'   # NEW cache — v1 cache is train+val only
MAX_TRAIN_PER_CLUSTER = 1500

if TEXT_EMB_PATH.exists():
    T = torch.load(TEXT_EMB_PATH, weights_only=False)
else:
    ds = load_dataset('google-research-datasets/go_emotions', 'simplified')
    names = ds['train'].features['labels'].feature.names
    to_c = lambda ids: C2I[EMOTION_TO_CLUSTER.get(names[ids[0]], 'Ambiguity')] if ids else None
    tok = AutoTokenizer.from_pretrained('roberta-large')
    rng = random.Random(42)
    T = {}
    for split in ['train', 'validation', 'test']:
        items = [(it['text'], to_c(it['labels'])) for it in ds[split]]
        items = [x for x in items if x[1] is not None]
        if split == 'train':   # balanced subsample, as in v1
            by_c = {c: [x for x in items if x[1] == c] for c in range(4)}
            items = [x for c in range(4) for x in rng.sample(by_c[c], min(MAX_TRAIN_PER_CLUSTER, len(by_c[c])))]
        texts = [x[0] for x in items]; labels = torch.tensor([x[1] for x in items])
        embs = []
        for i in range(0, len(texts), 32):
            enc = tok(texts[i:i+32], padding='max_length', truncation=True, max_length=128, return_tensors='pt')
            with torch.no_grad(), torch.cuda.amp.autocast():
                out = text_model.encoder.encoder(input_ids=enc['input_ids'].to(device),
                                                 attention_mask=enc['attention_mask'].to(device))
            embs.append(out.last_hidden_state[:, 0, :].cpu().float())
        T[split] = {'emb': torch.cat(embs), 'cluster': labels, 'texts': texts}
        print(f'{split}: {len(texts)} texts embedded', flush=True)
    torch.save(T, TEXT_EMB_PATH)

with torch.no_grad():
    for split in T:
        T[split]['probs'] = F.softmax(text_model.classifier.classifier(T[split]['emb'].to(device)).float(), -1).cpu()
from sklearn.metrics import accuracy_score, f1_score
y = T['test']['cluster']; p = T['test']['probs'].argmax(-1)
print(f"text test (full GoEmotions test, n={len(y)}): acc {accuracy_score(y, p):.4f} | macro-F1 {f1_score(y, p, average='macro'):.4f}")
print('→ should reproduce Stage 1: acc 0.7076 | macro-F1 0.6822')

## 5b · REAL Stage 1 + Stage 2 per-class reports & confusion matrices
Replaces the hand-typed Tables 6/7 and Figs. 12/14 in the paper.
Stage 1 = full GoEmotions test partition. Stage 2 = RAVDESS actors 22–24.
(Check the overall acc printed here matches 0.7076 / ~0.4056 — if Stage 2 differs slightly, it is because
this cell uses the Stage 3 audio preprocessing; report whichever the paper states and say so.)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt

def report(name, y, p, fname):
    print(f'═════ {name} ═════  n={len(y)}')
    print(classification_report(y, p, target_names=CLUSTER_NAMES, digits=4))
    cm = confusion_matrix(y, p)
    print('Confusion matrix (rows = true, cols = predicted):'); print(cm)
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(cm, cmap='Blues')
    ax.set_xticks(range(4)); ax.set_yticks(range(4))
    ax.set_xticklabels(CLUSTER_NAMES, rotation=20); ax.set_yticklabels(CLUSTER_NAMES)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True'); ax.set_title(f'{name} — confusion matrix (n={len(y)})')
    for i in range(4):
        for j in range(4):
            ax.text(j, i, cm[i, j], ha='center', va='center', color='white' if cm[i, j] > cm.max() / 2 else 'black')
    plt.colorbar(im, ax=ax, fraction=0.04); plt.tight_layout()
    plt.savefig(OUT_DIR / fname, dpi=200); plt.show()
    return {'report': classification_report(y, p, target_names=CLUSTER_NAMES, digits=4, output_dict=True),
            'confusion': cm.tolist(), 'n': int(len(y))}

STAGE_REPORTS = {
    'stage1_text_goemotions_test': report('Stage 1 (text)', T['test']['cluster'].numpy(),
                                          T['test']['probs'].argmax(-1).numpy(), 'stage1_confusion_real.png'),
    'stage2_audio_actors22_24': report('Stage 2 (audio)', audio_clusters[a_split['test']].numpy(),
                                       audio_probs[a_split['test']].argmax(-1).numpy(), 'stage2_confusion_real.png'),
}

## 6 · Leakage-free pairs (text partition × actor split)

In [ ]:
def build_pairs(text_split, audio_mask, seed, congruent=True):
    """Every text in the partition gets one random audio clip from the allowed actors:
    same cluster (congruent) or a different cluster (incongruent)."""
    r = random.Random(seed)
    pool = {c: [i for i in torch.nonzero(audio_mask).squeeze(-1).tolist() if audio_clusters[i].item() == c] for c in range(4)}
    pairs = []
    for ti, c in enumerate(T[text_split]['cluster'].tolist()):
        ac = c if congruent else r.choice([k for k in range(4) if k != c])
        pairs.append((ti, r.choice(pool[ac]), c, ac))
    return pairs

PAIRS = {'train': build_pairs('train', a_split['train'], 1),
         'val':   build_pairs('validation', a_split['val'], 2),
         'test':  build_pairs('test', a_split['test'], 3)}
for k, v in PAIRS.items(): print(k, len(v), 'pairs')
SPLIT_TEXT = {'train': 'train', 'val': 'validation', 'test': 'test'}

def tensors(split, pairs=None):
    pairs = pairs or PAIRS[split]; ts = SPLIT_TEXT[split]
    ti = torch.tensor([p[0] for p in pairs]); ai = torch.tensor([p[1] for p in pairs])
    return (T[ts]['emb'][ti], audio_embs[ai], torch.tensor([p[2] for p in pairs]),
            T[ts]['probs'][ti], audio_probs[ai])

## 7 · Single-stream and late-fusion baselines on the SAME test pairs

In [ ]:
def metrics(y, p): return {'acc': float(accuracy_score(y, p)), 'f1': float(f1_score(y, p, average='macro'))}
_, _, y_test, pt_test, pa_test = tensors('test')
BASE = {'text_only': metrics(y_test, pt_test.argmax(-1)),
        'audio_only': metrics(y_test, pa_test.argmax(-1)),
        'late_fusion_avg': metrics(y_test, (pt_test + pa_test).argmax(-1))}
late_pred = (pt_test + pa_test).argmax(-1)
for k, v in BASE.items(): print(f"{k:16s} acc {v['acc']:.4f} | F1 {v['f1']:.4f}")

## 8 · CMAF + ablation variants

In [ ]:
class CMAF(nn.Module):
    def __init__(self, d=256, heads=4, dropout=0.1, attention=True, aux=True):
        super().__init__()
        self.attention, self.aux = attention, aux
        self.text_proj, self.audio_proj = nn.Linear(1024, d), nn.Linear(1024, d)
        if attention:
            self.t2a = nn.MultiheadAttention(d, heads, dropout=dropout, batch_first=True)
            self.a2t = nn.MultiheadAttention(d, heads, dropout=dropout, batch_first=True)
            self.norm_t, self.norm_a = nn.LayerNorm(d), nn.LayerNorm(d)
        self.fuse = nn.Sequential(nn.Linear(2 * d, d), nn.GELU(), nn.Dropout(dropout), nn.Linear(d, 4))
        self.head_text, self.head_audio = nn.Linear(d, 4), nn.Linear(d, 4)
    def forward(self, et, ea):
        t, a = self.text_proj(et).unsqueeze(1), self.audio_proj(ea).unsqueeze(1)
        if self.attention:
            ta, _ = self.t2a(t, a, a); at, _ = self.a2t(a, t, t)
            t, a = self.norm_t(t + ta), self.norm_a(a + at)
        t, a = t.squeeze(1), a.squeeze(1)
        return {'fused_logits': self.fuse(torch.cat([t, a], -1)),
                'text_logits': self.head_text(t), 'audio_logits': self.head_audio(a)}

def jsd(p, q, eps=1e-8):
    m = 0.5 * (p + q)
    kl = lambda x: (x * (x.clamp_min(eps).log2() - m.clamp_min(eps).log2())).sum(-1)
    return 0.5 * (kl(p) + kl(q))

CONFIGS = {'full': {}, 'concat': {'attention': False}, 'no_aux': {'aux': False},
           'd128': {'d': 128}, 'd512': {'d': 512}}
SEEDS = [42, 43, 44, 45, 46]
TR, VA, TE = tensors('train'), tensors('val'), tensors('test')

def predict(model, et, ea):
    model.eval()
    with torch.no_grad(): return model(et.to(device), ea.to(device))

def train_one(cfg, seed, epochs=10, lr=3e-4, alpha=0.3, bs=64):
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
    model = CMAF(**cfg).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    et, ea, y = TR[0], TR[1], TR[2]
    w = (1.0 / torch.bincount(y, minlength=4).float())[y]
    best, best_state = -1, None
    for ep in range(epochs):
        model.train()
        idx = torch.multinomial(w, len(y), replacement=True)
        for i in range(0, len(idx), bs):
            b = idx[i:i + bs]
            out = model(et[b].to(device), ea[b].to(device)); yb = y[b].to(device)
            loss = F.cross_entropy(out['fused_logits'], yb)
            if model.aux:
                loss = loss + alpha * (F.cross_entropy(out['text_logits'], yb) + F.cross_entropy(out['audio_logits'], yb))
            opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        vf1 = f1_score(VA[2], predict(model, VA[0], VA[1])['fused_logits'].argmax(-1).cpu(), average='macro')
        if vf1 > best: best, best_state = vf1, {k: v.clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    return model, best

RESULTS, MODELS = {}, {}
for name, cfg in CONFIGS.items():
    runs = []
    for s in SEEDS:
        m, vf1 = train_one(cfg, s)
        pred = predict(m, TE[0], TE[1])['fused_logits'].argmax(-1).cpu()
        runs.append({'seed': s, 'val_f1': float(vf1), **metrics(TE[2], pred)})
        if name == 'full' and s == 42: MODELS['full'], full_pred = m, pred
    accs = np.array([r['acc'] for r in runs]); f1s = np.array([r['f1'] for r in runs])
    RESULTS[name] = {'runs': runs, 'acc_mean': float(accs.mean()), 'acc_std': float(accs.std(ddof=1)),
                     'f1_mean': float(f1s.mean()), 'f1_std': float(f1s.std(ddof=1)),
                     'params': sum(p.numel() for p in CMAF(**cfg).parameters())}
    print(f"{name:7s} acc {accs.mean()*100:.2f} ± {accs.std(ddof=1)*100:.2f} | F1 {f1s.mean():.4f} ± {f1s.std(ddof=1):.4f} | params {RESULTS[name]['params']:,}", flush=True)

## 9 · Significance: bootstrap CI + McNemar (CMAF full, seed 42)

In [ ]:
from scipy.stats import binomtest
from sklearn.metrics import classification_report, confusion_matrix
yt = TE[2].numpy(); fp = full_pred.numpy(); lp = late_pred.numpy(); tp = pt_test.argmax(-1).numpy()
rng = np.random.default_rng(0); n = len(yt)
boot = [accuracy_score(yt[i], fp[i]) for i in (rng.integers(0, n, n) for _ in range(2000))]
ci = (float(np.percentile(boot, 2.5)), float(np.percentile(boot, 97.5)))

def mcnemar(a, b):
    ca, cb = (a == yt), (b == yt)
    b01, b10 = int((ca & ~cb).sum()), int((~ca & cb).sum())
    return {'cmaf_only_correct': b01, 'other_only_correct': b10,
            'p_value': float(binomtest(b01, b01 + b10, 0.5).pvalue) if b01 + b10 else 1.0}

SIG = {'full_seed42_acc': float(accuracy_score(yt, fp)), 'bootstrap_95ci': ci,
       'mcnemar_vs_late_fusion': mcnemar(fp, lp), 'mcnemar_vs_text_only': mcnemar(fp, tp)}
print(json.dumps(SIG, indent=1))
PER_CLASS = classification_report(yt, fp, target_names=CLUSTER_NAMES, digits=4, output_dict=True)
print(classification_report(yt, fp, target_names=CLUSTER_NAMES, digits=4))
CONF = confusion_matrix(yt, fp).tolist(); print(np.array(CONF))

## 10 · JSD evaluation — test actors only
*Congruent* = text paired with same-cluster audio; *incongruent* = text paired with different-cluster audio.
Both use GoEmotions **test** text and RAVDESS actors **22–24** only.
- **standalone** = Stage 1 vs Stage 2 posteriors (what the deployed app uses)
- **aux** = CMAF auxiliary heads (what v1 / Table 10 measured)

In [ ]:
from sklearn.metrics import roc_auc_score
def jsd_scores(split, congruent, seed):
    pairs = build_pairs(SPLIT_TEXT[split], a_split[split], seed, congruent)
    et, ea, y, pt, pa = tensors(split, pairs)
    out = predict(MODELS['full'], et, ea)
    aux = jsd(F.softmax(out['text_logits'], -1), F.softmax(out['audio_logits'], -1)).cpu()
    std = jsd(pt, pa)
    mismatch = (pt.argmax(-1) != pa.argmax(-1))
    return {'standalone': std.numpy(), 'aux': aux.numpy(), 'mismatch': mismatch.numpy()}

JSD = {}
for split, seeds in [('val', (10, 11)), ('test', (20, 21))]:
    JSD[split] = {'cong': jsd_scores(split, True, seeds[0]), 'incong': jsd_scores(split, False, seeds[1])}

JSD_RES = {}
for kind in ['standalone', 'aux']:
    c, i = JSD['test']['cong'][kind], JSD['test']['incong'][kind]
    y = np.r_[np.zeros(len(c)), np.ones(len(i))]; s = np.r_[c, i]
    # threshold calibrated on VAL (Youden J), applied to TEST
    vc, vi = JSD['val']['cong'][kind], JSD['val']['incong'][kind]
    grid = np.unique(np.r_[vc, vi]); best_t, best_j = 0.5, -1
    for t in grid:
        j = (vi > t).mean() - (vc > t).mean()
        if j > best_j: best_j, best_t = j, float(t)
    JSD_RES[kind] = {'cong_mean': float(c.mean()), 'cong_std': float(c.std()), 'cong_median': float(np.median(c)),
                     'incong_mean': float(i.mean()), 'incong_std': float(i.std()), 'incong_median': float(np.median(i)),
                     'ratio': float(i.mean() / max(c.mean(), 1e-9)), 'roc_auc': float(roc_auc_score(y, s)),
                     'tau_val_calibrated': best_t,
                     'test_detect_rate@tau': float((i > best_t).mean()), 'test_false_alarm@tau': float((c > best_t).mean()),
                     'test_detect_rate@0.5': float((i > 0.5).mean()), 'test_false_alarm@0.5': float((c > 0.5).mean())}

mc, mi = JSD['test']['cong']['mismatch'], JSD['test']['incong']['mismatch']
sc, si = JSD['test']['cong']['standalone'], JSD['test']['incong']['standalone']
t = JSD_RES['standalone']['tau_val_calibrated']
JSD_RES['alert_rule_deployed_tau0.5'] = {'detect_rate': float((mi | (si > 0.5)).mean()), 'false_alarm': float((mc | (sc > 0.5)).mean())}
JSD_RES['alert_rule_calibrated_tau'] = {'tau': t, 'detect_rate': float((mi | (si > t)).mean()), 'false_alarm': float((mc | (sc > t)).mean())}
JSD_RES['mismatch_only'] = {'detect_rate': float(mi.mean()), 'false_alarm': float(mc.mean())}
print(json.dumps(JSD_RES, indent=1))

import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
for k, kind in enumerate(['standalone', 'aux']):
    ax[k].hist(JSD['test']['cong'][kind], bins=40, alpha=.6, label='congruent')
    ax[k].hist(JSD['test']['incong'][kind], bins=40, alpha=.6, label='incongruent')
    ax[k].set_title(f"JSD ({kind}) — AUC {JSD_RES[kind]['roc_auc']:.3f}"); ax[k].set_xlabel('JSD'); ax[k].legend()
plt.tight_layout(); plt.savefig(OUT_DIR / 'jsd_v2.png', dpi=200); plt.show()

## 12 · Calibration (ECE), attention check, error analysis

In [ ]:
def ece(probs, y, n_bins=15):
    conf, pred = probs.max(-1); acc = (pred == y).float(); e = 0.0
    edges = torch.linspace(0, 1, n_bins + 1)
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any(): e += m.float().mean().item() * abs(acc[m].mean().item() - conf[m].mean().item())
    return float(e)

out = predict(MODELS['full'], TE[0], TE[1])
p_fused = F.softmax(out['fused_logits'], -1).float().cpu()
ECE = {'text_only': ece(pt_test, y_test), 'audio_only': ece(pa_test, y_test),
       'late_fusion_avg': ece((pt_test + pa_test) / 2, y_test), 'cmaf_full_seed42': ece(p_fused, y_test)}
print('ECE:', json.dumps(ECE, indent=1))

# reliability diagram
fig, ax = plt.subplots(figsize=(5, 5))
for name, pr in [('text-only', pt_test), ('late fusion', (pt_test + pa_test) / 2), ('CMAF', p_fused)]:
    conf, pred = pr.max(-1); acc = (pred == y_test).float()
    edges = torch.linspace(0, 1, 11); xs, ys = [], []
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.sum() > 20: xs.append(conf[m].mean().item()); ys.append(acc[m].mean().item())
    ax.plot(xs, ys, 'o-', label=f"{name} (ECE {ece(pr, y_test):.3f})")
ax.plot([0, 1], [0, 1], 'k--', lw=1); ax.set_xlabel('Confidence'); ax.set_ylabel('Accuracy'); ax.legend()
ax.set_title('Reliability diagram (test pairs)'); plt.tight_layout(); plt.savefig(OUT_DIR / 'reliability.png', dpi=200); plt.show()

# attention check: with one token per modality, softmax over a length-1 key is identically 1
m = MODELS['full']; m.eval()
with torch.no_grad():
    t = m.text_proj(TE[0][:256].to(device)).unsqueeze(1); a = m.audio_proj(TE[1][:256].to(device)).unsqueeze(1)
    _, w = m.t2a(t, a, a, need_weights=True)
ATTN = {'min_weight': float(w.min()), 'max_weight': float(w.max()), 'shape': list(w.shape)}
print('attention weights:', ATTN)

# error analysis: CMAF (seed 42) errors on test pairs
test_pairs = PAIRS['test']; texts = T['test']['texts']
fp_ = p_fused.argmax(-1); tp_ = pt_test.argmax(-1); ap_ = pa_test.argmax(-1)
err = (fp_ != y_test).nonzero().squeeze(-1).tolist()
ERR = {'n_errors': len(err),
       'errors_where_text_correct': int(sum(int(tp_[i] == y_test[i]) for i in err)),
       'errors_where_audio_correct': int(sum(int(ap_[i] == y_test[i]) for i in err)),
       'errors_where_both_wrong': int(sum(int(tp_[i] != y_test[i] and ap_[i] != y_test[i]) for i in err)),
       'fixed_by_fusion_vs_text': int(((fp_ == y_test) & (tp_ != y_test)).sum()),
       'broken_by_fusion_vs_text': int(((fp_ != y_test) & (tp_ == y_test)).sum())}
r = random.Random(0); sample = r.sample(err, min(25, len(err)))
ERR['examples'] = [{'text': texts[test_pairs[i][0]], 'true': CLUSTER_NAMES[int(y_test[i])],
                    'cmaf': CLUSTER_NAMES[int(fp_[i])], 'text_only': CLUSTER_NAMES[int(tp_[i])],
                    'audio_only': CLUSTER_NAMES[int(ap_[i])]} for i in sample]
print(json.dumps(ERR, indent=1, ensure_ascii=False))

## 11 · Save + SUMMARY (copy this block to Claude)

In [ ]:
ALL = {'ece': ECE, 'attention_check': ATTN, 'error_analysis': ERR, 'stage_reports': STAGE_REPORTS, 'baselines': BASE, 'cmaf': RESULTS, 'significance': SIG, 'per_class_full_seed42': PER_CLASS,
       'confusion_full_seed42': CONF, 'jsd': JSD_RES,
       'n_pairs': {k: len(v) for k, v in PAIRS.items()}}
with open(OUT_DIR / 'results_stage3_v2.json', 'w') as f: json.dump(ALL, f, indent=1)
torch.save({'model_state_dict': MODELS['full'].state_dict(), 'note': 'Stage3 v2, seed 42, leakage-free'},
           OUT_DIR / 'fusion_v2_seed42.pt')
print('===== SUMMARY START =====')
print(json.dumps(ALL, indent=1))
print('===== SUMMARY END =====')